In [ ]:
%load_ext dotenv
%dotenv .env

# Helper Functions

In [ ]:
import time
from pydantic import BaseModel, Field

from redbox.chains.parser import StreamingJsonOutputParser, BaseMessage
from typing import List, Union
import matplotlib.pyplot as plt
import numpy as np


class MetricsCollector:
    def __init__(self):
        self.tokens = 0
        self.timestamps = []

    def record_tokens(self, n_tokens):
        now = time.perf_counter()
        self.tokens += n_tokens
        self.timestamps.append((now, self.tokens))

    def compute_throughput(self):
        """Cumulative tokens/sec"""
        if not self.timestamps:
            return []
        start_time = self.timestamps[0][0]
        throughput = [
            (t - start_time, tokens / (t - start_time) if (t - start_time) > 0 else 0) for t, tokens in self.timestamps
        ]
        return throughput

    def compute_instantaneous_throughput(self):
        """Instantaneous tokens/sec per chunk"""
        if len(self.timestamps) < 2:
            return []
        throughput = []
        for i in range(1, len(self.timestamps)):
            t_prev, tokens_prev = self.timestamps[i - 1]
            t_curr, tokens_curr = self.timestamps[i]
            dt = t_curr - t_prev
            d_tokens = tokens_curr - tokens_prev
            tps = d_tokens / dt if dt > 0 else 0
            throughput.append((t_curr - self.timestamps[0][0], tps))
        return throughput


class StreamingJsonOutputParserWithMetrics(StreamingJsonOutputParser):
    metrics: MetricsCollector = Field(default_factory=MetricsCollector)

    class Config:
        arbitrary_types_allowed = True  # allow MetricsCollector type
        extra = "allow"  # allow other attributes


class DummySchema(BaseModel):
    answer: str
    citations: list = []


def generate_large_input(num_chunks: int = 100, chunk_size: int = 10) -> List[str]:
    base_text = "Hello world, this is a streaming benchmark test. "
    text = base_text * num_chunks

    full_json = f'```{{"answer": "{text}"}}```'

    chunks = []
    for i in range(0, len(full_json), chunk_size):
        chunks.append(full_json[i : i + chunk_size])

    return chunks


# def benchmark_streaming(parser: StreamingJsonOutputParserWithMetrics, input_chunks: List[Union[str, BaseMessage]]):
#     for parsed in parser._transform(input_chunks):
#         pass

#     return parser.metrics


# def benchmark_multiple_runs(parser_class, input_chunks, num_runs: int = 5, **parser_kwargs):
#     """Run parser multiple times and collect metrics."""
#     all_metrics = []

#     for run_idx in range(num_runs):
#         parser = parser_class(**parser_kwargs)
#         metrics = benchmark_streaming(parser, input_chunks)
#         all_metrics.append(metrics)

#     return all_metrics


def aggregate_metrics_with_chunks(metrics_list: List[MetricsCollector]):
    """
    Interpolate and compute mean/std for tokens/sec and chunks/sec (cumulative and instantaneous)
    """
    max_time = max(m.timestamps[-1][0] - m.timestamps[0][0] for m in metrics_list if m.timestamps)
    common_times = np.linspace(0, max_time, 1000)

    # Tokens
    cum_tokens_runs, inst_tokens_runs = [], []
    cum_chunks_runs, inst_chunks_runs = [], []

    for m in metrics_list:
        if not m.timestamps:
            continue
        # Cumulative tokens
        t_times, cum_tokens = zip(*m.compute_throughput())
        t_times = np.array(t_times) - t_times[0]
        cum_tokens_interp = np.interp(common_times, t_times, cum_tokens)
        cum_tokens_runs.append(cum_tokens_interp)

        # Instantaneous tokens
        inst_tokens_data = m.compute_instantaneous_throughput()
        if inst_tokens_data:
            inst_times, inst_tokens = zip(*inst_tokens_data)
            inst_times = np.array(inst_times) - inst_times[0]
            inst_tokens_interp = np.interp(common_times, inst_times, inst_tokens, left=0, right=0)
        else:
            inst_tokens_interp = np.zeros_like(common_times)
        inst_tokens_runs.append(inst_tokens_interp)

        # # Cumulative chunks
        # t_times, cum_chunks = zip(*m.compute_chunk_throughput())
        # t_times = np.array(t_times) - t_times[0]
        # cum_chunks_interp = np.interp(common_times, t_times, cum_chunks)
        # cum_chunks_runs.append(cum_chunks_interp)

        # # Instantaneous chunks
        # inst_chunks_data = m.compute_instantaneous_chunk_throughput()
        # if inst_chunks_data:
        #     inst_times, inst_chunks = zip(*inst_chunks_data)
        #     inst_times = np.array(inst_times) - inst_times[0]
        #     inst_chunks_interp = np.interp(common_times, inst_times, inst_chunks, left=0, right=0)
        # else:
        #     inst_chunks_interp = np.zeros_like(common_times)
        # inst_chunks_runs.append(inst_chunks_interp)

    cum_tokens_runs = np.array(cum_tokens_runs)
    inst_tokens_runs = np.array(inst_tokens_runs)
    cum_chunks_runs = np.array(cum_chunks_runs)
    inst_chunks_runs = np.array(inst_chunks_runs)

    return (
        common_times,
        np.mean(cum_tokens_runs, axis=0),
        np.std(cum_tokens_runs, axis=0),
        np.mean(inst_tokens_runs, axis=0),
        np.std(inst_tokens_runs, axis=0),
        np.mean(cum_chunks_runs, axis=0),
        np.std(cum_chunks_runs, axis=0),
        np.mean(inst_chunks_runs, axis=0),
        np.std(inst_chunks_runs, axis=0),
    )


def plot_aggregate_tokens_and_chunks(agg_data_list, labels):
    fig, axes = plt.subplots(1, 2, figsize=(14, 10))
    axes = axes.flatten()

    for agg_data, label in zip(agg_data_list, labels):
        (
            times,
            mean_cum_tokens,
            std_cum_tokens,
            mean_inst_tokens,
            std_inst_tokens,
            mean_cum_chunks,
            std_cum_chunks,
            mean_inst_chunks,
            std_inst_chunks,
        ) = agg_data

        # Cumulative tokens
        axes[0].plot(times, mean_cum_tokens, label=label)
        axes[0].fill_between(times, mean_cum_tokens - std_cum_tokens, mean_cum_tokens + std_cum_tokens, alpha=0.3)
        axes[0].set_title("Cumulative Throughput (Tokens/sec)")
        # axes[0].set_yscale("log")
        axes[0].grid(True)

        # Instantaneous tokens
        axes[1].plot(times, mean_inst_tokens, label=label)
        axes[1].fill_between(times, mean_inst_tokens - std_inst_tokens, mean_inst_tokens + std_inst_tokens, alpha=0.3)
        axes[1].set_title("Instantaneous Throughput per Chunk (Tokens/sec)")
        # axes[1].set_yscale("log")
        axes[1].grid(True)

        # # Cumulative chunks
        # axes[2].plot(times, mean_cum_chunks, label=label)
        # axes[2].fill_between(times, mean_cum_chunks - std_cum_chunks, mean_cum_chunks + std_cum_chunks, alpha=0.3)
        # axes[2].set_title("Cumulative Throughput (Chunks/sec)")
        # # axes[2].set_yscale("log")
        # axes[2].grid(True)

        # # Instantaneous chunks
        # axes[3].plot(times, mean_inst_chunks, label=label)
        # axes[3].fill_between(times, mean_inst_chunks - std_inst_chunks, mean_inst_chunks + std_inst_chunks, alpha=0.3)
        # axes[3].set_title("Instantaneous Throughput per Chunk (Chunks/sec)")
        # # axes[3].set_yscale("log")
        # axes[3].grid(True)

    for ax in axes:
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Throughput/sec")
        ax.legend()

    plt.tight_layout()
    plt.show()

# Sync Parser

In [ ]:
import re
from typing import List, Iterator, Any

from langchain_core.outputs import ChatGenerationChunk, GenerationChunk

# from tests.parser.plot_util import (
#     StreamingJsonOutputParserWithMetrics,
#     DummySchema,
#     generate_large_input,
#     aggregate_metrics_with_chunks,
#     plot_aggregate_tokens_and_chunks,
# )


# - Parser Variants [Sync]


class StreamingJsonOriginal(StreamingJsonOutputParserWithMetrics):
    def extract_json(self, text):
        if isinstance(text, list):
            text = text[0].get("text")

        try:
            # Find content between first { and last }
            match = re.search(r"(\{.*\})", text, re.DOTALL)
            if not match:
                return text

            json_str = match.group(1)

            return json_str

        except Exception as e:
            print(f"Error processing JSON: {e}")
            return text

    def _transform(self, input: Iterator[Union[str, BaseMessage]]):
        acc_gen: Union[GenerationChunk, ChatGenerationChunk, None] = None
        field_length_at_last_run: int = 0
        parsed = None
        is_parsed = False
        for chunk in input:
            chunk_gen = self._to_generation_chunk(chunk)
            acc_gen = chunk_gen if acc_gen is None else acc_gen + chunk_gen  # type: ignore[operator]
            if parsed := self.parse_partial_json(acc_gen.text):
                is_parsed = True
                field_content = parsed.get(self.name_of_streamed_field)
                if field_content:
                    if new_tokens := field_content[field_length_at_last_run:]:
                        # parser_module.dispatch_custom_event(RedboxEventType.response_tokens, data=new_tokens)
                        self.metrics.record_tokens(len(new_tokens))
                        field_length_at_last_run = len(field_content)
                        yield self.pydantic_schema_object.model_validate(parsed)
        if not is_parsed:  # if no tokens were parsed, parse last chunk
            match = re.search(r"(\{)", acc_gen.text, re.DOTALL)
            if not match:  # stream only when text does not contain json brackets to ensure quality of output
                transformed_text = self.answer_str_to_json(acc_gen.text)
                if parsed := self.parse_partial_json(transformed_text):
                    field_content = parsed.get(self.name_of_streamed_field)
                    if field_content:
                        # parser_module.dispatch_custom_event(RedboxEventType.response_tokens, data=field_content)
                        self.metrics.record_tokens(len(field_content))
                        yield self.pydantic_schema_object.model_validate(parsed)

        if parsed:
            yield self.pydantic_schema_object.model_validate(parsed)


class StreamingJsonRefactored(StreamingJsonOutputParserWithMetrics):
    def extract_json(self, text: str) -> str:
        """Extract JSON from text more efficiently."""
        if isinstance(text, list):
            text = text[0].get("text")

        # Find first { and last } instead of using a greedy regex
        start_idx = text.find("{")
        end_idx = text.rfind("}")
        if start_idx == -1 or end_idx == -1 or end_idx <= start_idx:
            return text  # no JSON found

        return text[start_idx : end_idx + 1]

    def _transform(self, input: Iterator[Union[str, BaseMessage]]) -> Iterator[Any]:
        acc_text = ""  # accumulated unparsed text
        field_length_at_last_run = 0
        parsed = None
        seen_json_start = False

        for chunk in input:
            chunk_gen = self._to_generation_chunk(chunk)
            text = chunk_gen.text
            acc_text += text  # append only new chunk

            if not seen_json_start:
                if "{" in text:
                    seen_json_start = True
                else:
                    # pre-JSON text, record immediately
                    self._record_tokens(text)
                    continue

            # parse only if new text added after last parse
            partial = self.parse_partial_json(acc_text)
            if not partial:
                continue

            parsed = partial
            field_content = parsed.get(self.name_of_streamed_field)
            if not field_content:
                continue

            # record only **new tokens**
            if len(field_content) > field_length_at_last_run:
                new_tokens = field_content[field_length_at_last_run:]
                self.metrics.record_tokens(len(new_tokens))
                field_length_at_last_run = len(field_content)
                yield self.pydantic_schema_object.model_validate(parsed)

        # final parse if anything remains
        if parsed is None and acc_text:
            if "{" not in acc_text:
                acc_text = self.answer_str_to_json(acc_text)

            if parsed := self.parse_partial_json(acc_text):
                if field_content := parsed.get(self.name_of_streamed_field):
                    self.metrics.record_tokens(len(field_content))
                    yield self.pydantic_schema_object.model_validate(parsed)

        if parsed:
            yield self.pydantic_schema_object.model_validate(parsed)


# - Run the sync parser benchmark


def benchmark_streaming(parser: StreamingJsonOutputParserWithMetrics, input_chunks: List[Union[str, BaseMessage]]):
    for parsed in parser._transform(input_chunks):
        pass

    return parser.metrics


def benchmark_multiple_runs(parser_class, input_chunks, num_runs: int = 5, **parser_kwargs):
    """Run parser multiple times and collect metrics."""
    all_metrics = []

    for _ in range(num_runs):
        parser = parser_class(**parser_kwargs)
        metrics = benchmark_streaming(parser, input_chunks)
        all_metrics.append(metrics)

    return all_metrics

In [ ]:
input_chunks = generate_large_input(num_chunks=200, chunk_size=10)

num_runs = 10
original_runs = benchmark_multiple_runs(
    StreamingJsonOriginal, input_chunks, num_runs=num_runs, pydantic_schema_object=DummySchema
)
refactored_runs = benchmark_multiple_runs(
    StreamingJsonRefactored, input_chunks, num_runs=num_runs, pydantic_schema_object=DummySchema
)

# Aggregate metrics
original_agg = aggregate_metrics_with_chunks(original_runs)
refactored_agg = aggregate_metrics_with_chunks(refactored_runs)

# Plot mean +- std.dev
plot_aggregate_tokens_and_chunks([original_agg, refactored_agg], ["Original", "Refactored"])

# Async Parser

In [ ]:
import asyncio
from typing import List, Union, Any, AsyncIterator

from redbox.chains.parser import BaseMessage

# - Parser Variants [Async]


class StreamingJsonOriginal(StreamingJsonOutputParserWithMetrics):
    def extract_json(self, text):
        if isinstance(text, list):
            text = text[0].get("text")

        try:
            # Find content between first { and last }
            match = re.search(r"(\{.*\})", text, re.DOTALL)
            if not match:
                return text

            json_str = match.group(1)

            return json_str

        except Exception as e:
            print(f"Error processing JSON: {e}")
            return text

    async def _atransform(self, input: AsyncIterator[Union[str, BaseMessage]]) -> AsyncIterator[Any]:
        acc_gen: Union[GenerationChunk, ChatGenerationChunk, None] = None
        field_length_at_last_run: int = 0
        parsed = None
        is_parsed = False

        async for chunk in input:
            chunk_gen = self._to_generation_chunk(chunk)

            acc_gen = chunk_gen if acc_gen is None else acc_gen + chunk_gen  # type: ignore[operator]
            if parsed := self.parse_partial_json(acc_gen.text):
                is_parsed = True
                field_content = parsed.get(self.name_of_streamed_field)
                if field_content:
                    if new_tokens := field_content[field_length_at_last_run:]:
                        self.metrics.record_tokens(len(new_tokens))
                        field_length_at_last_run = len(field_content)
                        yield self.pydantic_schema_object.model_validate(parsed)

        if not is_parsed:  # if no tokens were parsed, parse last chunk
            match = re.search(r"(\{)", acc_gen.text, re.DOTALL)
            if not match:  # stream only when text does not contain json brackets to ensure quality of output
                transformed_text = self.answer_str_to_json(acc_gen.text)
                if parsed := self.parse_partial_json(transformed_text):
                    field_content = parsed.get(self.name_of_streamed_field)
                    if field_content:
                        self.metrics.record_tokens(len(field_content))

                        yield self.pydantic_schema_object.model_validate(parsed)

        if parsed:
            yield self.pydantic_schema_object.model_validate(parsed)


class StreamingJsonRefactored(StreamingJsonOutputParserWithMetrics):
    def extract_json(self, text: str) -> str:
        """Extract JSON from text more efficiently."""
        if isinstance(text, list):
            text = text[0].get("text")

        start_idx = text.find("{")
        end_idx = text.rfind("}")
        if start_idx == -1 or end_idx == -1 or end_idx <= start_idx:
            return text  # no JSON found

        return text[start_idx : end_idx + 1]

    async def _atransform(self, input: AsyncIterator[Union[str, BaseMessage]]) -> AsyncIterator[Any]:
        acc_buffer: list[str] = []
        field_length_at_last_run: int = 0
        parsed = None
        is_parsed = False
        seen_json_start = False

        async for chunk in input:
            chunk_gen = self._to_generation_chunk(chunk)
            text = chunk_gen.text

            acc_buffer.append(text)

            if not seen_json_start:
                if "{" not in text:
                    continue
                seen_json_start = True

            acc_text = "".join(acc_buffer)

            partial = self.parse_partial_json(acc_text)
            if not partial:
                continue

            is_parsed = True
            parsed = partial

            field_content = parsed.get(self.name_of_streamed_field)
            if not field_content:
                continue

            if len(field_content) > field_length_at_last_run:
                new_tokens = field_content[field_length_at_last_run:]
                self.metrics.record_tokens(len(new_tokens))
                field_length_at_last_run = len(field_content)
                yield self.pydantic_schema_object.model_validate(parsed)

        if not is_parsed:
            acc_text = "".join(acc_buffer)
            if "{" not in acc_text:
                acc_text = self.answer_str_to_json(acc_text)

            if parsed := self.parse_partial_json(acc_text):
                if field_content := parsed.get(self.name_of_streamed_field):
                    self.metrics.record_tokens(len(field_content))
                    yield self.pydantic_schema_object.model_validate(parsed)

        if parsed:
            yield self.pydantic_schema_object.model_validate(parsed)


class StreamingJsonOptimised(StreamingJsonOutputParserWithMetrics):
    def _find_answer_start(self, buffer: str, field_name: str = "answer") -> int:
        key = f'"{field_name}"'
        key_pos = buffer.find(key)
        if key_pos == -1:
            return -1
        colon_pos = buffer.find(":", key_pos + len(key))
        if colon_pos == -1:
            return -1
        quote_pos = buffer.find('"', colon_pos + 1)
        if quote_pos == -1:
            return -1
        return quote_pos + 1

    def _extract_answer_incremental(self, buffer: str, answer_start: int, resume_raw_pos: int) -> tuple[str, int]:
        escape_map = {
            "n": "\n",
            "t": "\t",
            "r": "\r",
            '"': '"',
            "\\": "\\",
            "b": "\b",
            "f": "\f",
        }
        i = resume_raw_pos if resume_raw_pos >= answer_start else answer_start
        result = []

        while i < len(buffer):
            c = buffer[i]
            if c == "\\":
                if i + 1 >= len(buffer):
                    break  # incomplete escape at boundary
                next_c = buffer[i + 1]
                if next_c in escape_map:
                    result.append(escape_map[next_c])
                    i += 2
                elif next_c == "u":
                    if i + 5 < len(buffer):
                        try:
                            codepoint = int(buffer[i + 2 : i + 6], 16)
                            result.append(chr(codepoint))
                            i += 6
                        except ValueError:
                            i += 1
                    else:
                        break  # incomplete \uXXXX
                else:
                    i += 1  # unknown escape, drop backslash
                continue
            if c == '"':
                return "".join(result), i
            result.append(c)
            i += 1

        return "".join(result), i

    async def _atransform(self, input: AsyncIterator) -> AsyncIterator[Any]:
        acc_text = ""
        acc_message = None
        field_length_at_last_run = 0
        is_parsed = False
        seen_json_start = False
        answer_start_pos = -1
        last_raw_pos = -1

        async for chunk in input:
            chunk_gen = self._to_generation_chunk(chunk)
            text = chunk_gen.text
            acc_text += text

            if acc_message is None:
                acc_message = chunk
            else:
                try:
                    acc_message = acc_message + chunk
                except Exception:
                    acc_message = chunk

            if not seen_json_start:
                if "{" not in acc_text:
                    continue
                seen_json_start = True

            # Fast path — answer field already located
            if answer_start_pos != -1:
                new_chars, last_raw_pos = self._extract_answer(acc_text, answer_start_pos, last_raw_pos)
                if new_chars:
                    self.metrics.record_tokens(len(new_chars))
                    field_length_at_last_run += len(new_chars)
                continue

            # Slow path — find answer field start
            answer_start_pos = self._find_answer_start(acc_text)
            if answer_start_pos != -1:
                last_raw_pos = answer_start_pos
                new_chars, last_raw_pos = self._extract_answer(acc_text, answer_start_pos, last_raw_pos)
                if new_chars:
                    is_parsed = True
                    self.metrics.record_tokens(len(new_chars))
                    field_length_at_last_run += len(new_chars)
                continue

            # Pre-answer preamble — full parse fallback
            try:
                partial = self.parse_partial_json(acc_text)
            except Exception:
                partial = None

            if partial:
                is_parsed = True
                field_content = partial.get(self.name_of_streamed_field)
                if field_content and len(field_content) > field_length_at_last_run:
                    new_tokens = field_content[field_length_at_last_run:]
                    self.metrics.record_tokens(len(new_tokens))
                    field_length_at_last_run = len(field_content)

        # End of stream — single full parse, single yield with citations
        if not is_parsed and "{" not in acc_text:
            acc_text = self.answer_str_to_json(acc_text)

        final_parsed = self.parse_partial_json(acc_text)
        if final_parsed:
            field_content = final_parsed.get(self.name_of_streamed_field)
            if field_content and field_length_at_last_run == 0:
                self.metrics.record_tokens(len(field_content))
            yield self.pydantic_schema_object.model_validate(final_parsed)

    # async def _atransform(self, input: AsyncIterator) -> AsyncIterator[Any]:
    #     acc_text = ""
    #     acc_message = None
    #     field_length_at_last_run = 0
    #     is_parsed = False
    #     seen_json_start = False
    #     answer_start_pos = -1
    #     last_raw_pos = -1

    #     async for chunk in input:
    #         chunk_gen = self._to_generation_chunk(chunk)
    #         text = chunk_gen.text
    #         acc_text += text

    #         if acc_message is None:
    #             acc_message = chunk
    #         else:
    #             try:
    #                 acc_message = acc_message + chunk
    #             except Exception:
    #                 acc_message = chunk

    #         if not seen_json_start:
    #             if "{" not in acc_text:
    #                 continue
    #             seen_json_start = True

    #         # Fast path — already found answer field
    #         if answer_start_pos != -1:
    #             new_chars, last_raw_pos = self._extract_answer_incremental(acc_text, answer_start_pos, last_raw_pos)
    #             if new_chars:
    #                 self.metrics.record_tokens(len(new_chars))
    #                 field_length_at_last_run += len(new_chars)
    #                 yield self.pydantic_schema_object.model_validate(
    #                     {self.name_of_streamed_field: acc_text[answer_start_pos:last_raw_pos], "citations": []}
    #                 )
    #             continue

    #         # Slow path — find answer field start
    #         answer_start_pos = self._find_answer_start(acc_text)
    #         if answer_start_pos != -1:
    #             last_raw_pos = answer_start_pos
    #             new_chars, last_raw_pos = self._extract_answer_incremental(acc_text, answer_start_pos, last_raw_pos)
    #             if new_chars:
    #                 is_parsed = True
    #                 self.metrics.record_tokens(len(new_chars))
    #                 field_length_at_last_run += len(new_chars)
    #                 yield self.pydantic_schema_object.model_validate(
    #                     {self.name_of_streamed_field: new_chars, "citations": []}
    #                 )
    #             continue

    #         # Pre-answer preamble — full parse fallback
    #         try:
    #             partial = self.parse_partial_json(acc_text)
    #         except Exception:
    #             partial = None

    #         if partial:
    #             is_parsed = True
    #             field_content = partial.get(self.name_of_streamed_field)
    #             if field_content and len(field_content) > field_length_at_last_run:
    #                 new_tokens = field_content[field_length_at_last_run:]
    #                 self.metrics.record_tokens(len(new_tokens))
    #                 field_length_at_last_run = len(field_content)
    #                 yield self.pydantic_schema_object.model_validate(partial)

    #     # End of stream — full parse for citations
    #     if not is_parsed and "{" not in acc_text:
    #         acc_text = self.answer_str_to_json(acc_text)

    #     final_parsed = self.parse_partial_json(acc_text)
    #     if final_parsed:
    #         field_content = final_parsed.get(self.name_of_streamed_field)
    #         if field_content and field_length_at_last_run == 0:
    #             self.metrics.record_tokens(len(field_content))
    #         yield self.pydantic_schema_object.model_validate(final_parsed)


# - Run the async parser


async def to_async_iter(chunks: List[Union[str, BaseMessage]]) -> AsyncIterator[Union[str, BaseMessage]]:
    for chunk in chunks:
        yield chunk
        await asyncio.sleep(0)  # allow event loop to switch if needed


async def abenchmark_streaming(
    parser: StreamingJsonOutputParserWithMetrics, input_chunks: List[Union[str, BaseMessage]]
):
    async_chunks = to_async_iter(input_chunks)

    async for parsed in parser._atransform(async_chunks):
        pass

    return parser.metrics


async def benchmark_multiple_runs(parser_class, input_chunks, num_runs: int = 5, **parser_kwargs):
    """Run parser multiple times and collect metrics."""
    all_metrics = []

    for _ in range(num_runs):
        parser = parser_class(**parser_kwargs)
        metrics = await abenchmark_streaming(parser, input_chunks)
        all_metrics.append(metrics)

    return all_metrics

In [ ]:
# - Run multiple benchmarks and aggregate


async def main():
    input_chunks = generate_large_input(num_chunks=200, chunk_size=10)

    num_runs = 10
    original_runs = await benchmark_multiple_runs(
        StreamingJsonOriginal, input_chunks, num_runs=num_runs, pydantic_schema_object=DummySchema
    )
    refactored_runs = await benchmark_multiple_runs(
        StreamingJsonRefactored, input_chunks, num_runs=num_runs, pydantic_schema_object=DummySchema
    )
    optimised_runs = await benchmark_multiple_runs(
        StreamingJsonOptimised, input_chunks, num_runs=num_runs, pydantic_schema_object=DummySchema
    )

    # Aggregate metrics
    original_agg = aggregate_metrics_with_chunks(original_runs)
    refactored_agg = aggregate_metrics_with_chunks(refactored_runs)
    optimised_agg = aggregate_metrics_with_chunks(optimised_runs)

    # Plot mean +- std.dev
    plot_aggregate_tokens_and_chunks(
        [original_agg, refactored_agg, optimised_agg], ["Original", "Refactored", "Optimised"]
    )


asyncio.run(main())